# Deep Learning Assignment 2
**Objective**: Implement a Single-Layer Perceptron for binary classification and analyze the effect of weights, bias, and activation functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Set random seed so the results are exactly reproducible
np.random.seed(42)

## 1. Create a Binary Classification Dataset
To test our Perceptron, we will generate a simple 2D dataset with two distinct groups of points. A single-layer perceptron can only solve problems that are **linearly separable** (meaning you can draw a straight line between the two classes).

In [ ]:
# Generate a linearly separable dataset with 100 samples
X, y = make_blobs(n_samples=100, centers=2, n_features=2, center_box=(0, 10), random_state=42)

# Visualize the dataset
plt.figure(figsize=(6, 4))
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='red', label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='blue', label='Class 1')
plt.title("Linearly Separable Binary Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 2. Implement the Single-Layer Perceptron (From Scratch)
Instead of using a pre-built library, implementing the Perceptron in raw Python (NumPy) helps us truly understand weights and biases.

**The Math:**
1. Calculate the linear combination: $z = (w_1 \cdot x_1) + (w_2 \cdot x_2) + b$
2. Pass it through an activation function (like a Step Function) to get the prediction (0 or 1).
3. Compare the prediction to the actual label and update the weights.

In [ ]:
class SingleLayerPerceptron:
    def __init__(self, learning_rate=0.01, epochs=100, activation='step'):
        self.lr = learning_rate
        self.epochs = epochs
        self.activation = activation
        self.weights = None
        self.bias = None
        
    def _activation_function(self, z):
        if self.activation == 'step':
            # Step function: Returns 1 if z >= 0, else 0
            return np.where(z >= 0, 1, 0)
        elif self.activation == 'sigmoid':
            # Sigmoid function: Returns a smooth curve between 0 and 1
            return 1 / (1 + np.exp(-z))
        else:
            raise ValueError("Activation must be 'step' or 'sigmoid'")

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize weights and bias to zeros
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Training loop
        for _ in range(self.epochs):
            for idx, x_i in enumerate(X):
                # 1. Forward pass (z = W*X + b)
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = self._activation_function(linear_output)
                
                # 2. Update rule
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        y_predicted = self._activation_function(linear_output)
        if self.activation == 'sigmoid':
            return np.where(y_predicted >= 0.5, 1, 0)
        return y_predicted

## 3. Train the Perceptron and Visualize the Decision Boundary
The Perceptron will try to draw a straight line (decision boundary) that perfectly separates Class 0 from Class 1.

In [ ]:
def plot_decision_boundary(X, y, model, title="Decision Boundary"):
    # Create a mesh grid to plot the colored regions
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))
    
    # Predict every point on the grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='red', edgecolor='k', label='Class 0')
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='blue', edgecolor='k', label='Class 1')
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()

# Train with Step Activation
perceptron = SingleLayerPerceptron(learning_rate=0.1, epochs=50, activation='step')
perceptron.fit(X, y)

print(f"Trained Weights: {perceptron.weights}")
print(f"Trained Bias: {perceptron.bias}")
plot_decision_boundary(X, y, perceptron, "Optimal Decision Boundary")

## 4. Analyze the Effect of Weights and Bias
The decision boundary is defined by the line equation: $w_1x_1 + w_2x_2 + b = 0$. 
* **Weights ($w_1, w_2$)** determine the *slope* (angle/rotation) of the decision boundary.
* **Bias ($b$)** determines the *position* (shift/translation) of the line away from the origin.

Let's manually mess with the learned bias and weights to see how the line breaks!

In [ ]:
import copy
model_experiment = copy.deepcopy(perceptron)

# 1. Effect of Changing the Bias (Shifts the line up/down/left/right without rotating it)
model_experiment.bias = perceptron.bias + 2.0  # Add 2.0 to the bias
plot_decision_boundary(X, y, model_experiment, "Effect of Increasing Bias (Line Shifts)")

# 2. Effect of Changing the Weights (Changes the slope/rotation of the line)
model_experiment.bias = perceptron.bias # Reset bias back to normal
model_experiment.weights[0] = perceptron.weights[0] * -1 # Flip the sign of the first weight
plot_decision_boundary(X, y, model_experiment, "Effect of Changing a Weight (Line Rotates)")

## 5. Analyze the Effect of Activation Functions
* **Step Function**: Outputs exactly `0` or `1`. It acts as a hard boundary. The network is strictly confident about its predictions, but it cannot express *uncertainty* (e.g., "I am 80% sure this is Class 1").
* **Sigmoid Function**: Outputs a continuous value between `0` and `1`. This acts like a probability curve. Points far away from the line are close to 100% confident (0.99 or 0.01), but points right on the boundary line sit at exactly `0.5` (50% uncertainty).

In [ ]:
# Visualize how the mathematical output differs between the two functions
z = np.linspace(-10, 10, 100)

step_output = np.where(z >= 0, 1, 0)
sigmoid_output = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, step_output, label='Step Function (Hard 0 or 1)', color='red', linestyle='dashed', linewidth=2)
plt.plot(z, sigmoid_output, label='Sigmoid Function (Smooth Probability)', color='blue', linewidth=2)
plt.title("Step vs Sigmoid Activation Functions")
plt.xlabel("Input (z = W*X + b)")
plt.ylabel("Activation Output")
plt.legend()
plt.grid(True)
plt.show()